# 2026-09-26 Public Data Center BOM / Architecture Re-validation

이 notebook은 `kimheeseo/LSCNS`의 현재 **shared v2 validation engine**을 Colab에서 직접 다시 실행하여 10개 공개 reference architecture의 정밀도를 재계산합니다.

**주의:** 이 10개는 기존 개발/회귀 benchmark에서 선정한 re-validation subset입니다. 따라서 결과는 regression/reproducibility accuracy이며, 새로운 unseen hold-out proof가 아닙니다.


## 평가 기준

- 각 metric absolute percentage error: `abs(calculated-reference)/abs(reference) × 100`
- PASS: 비교 가능한 각 metric error `< 10%`
- Case MAPE: case 내 비교 metric error의 평균
- Metric-weighted MAPE: 10개 case 전체 metric error를 한꺼번에 평균
- Reference가 공개하지 않은 물리 BOM 항목은 추측하지 않고 scoring에서 제외


In [ ]:
import os, json, shutil, subprocess, pathlib, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_DIR = pathlib.Path('/content/LSCNS')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','-q','https://github.com/kimheeseo/LSCNS.git',str(REPO_DIR)], check=True)
if shutil.which('node') is None:
    subprocess.run(['apt-get','-qq','update'], check=True)
    subprocess.run(['apt-get','-qq','install','-y','nodejs'], check=True)

BASE = REPO_DIR / 'DCI' / 'ver.1,2_reference_cases'
print('Repo:', REPO_DIR)
print('Node:', subprocess.check_output(['node','--version'], text=True).strip())


In [ ]:
CASES = [
 '01_google_tpu_v4',
 '03_google_tpu_v6e',
 '07_meta_rsc_phase1',
 '11_bytedance_megascale',
 '21_nvidia_dgx_h100_superpod',
 '22_nvidia_dgx_b200_superpod',
 '24_nvidia_gb200_nvl72',
 '29_frontier',
 '30_aurora',
 '31_nvidia_dgx_b300_1su',
]

SOURCE_MATRIX = [
 ('01_google_tpu_v4','Google TPU v4','Google','https://arxiv.org/abs/2304.01433'),
 ('03_google_tpu_v6e','Google TPU v6e / Trillium','Google','https://docs.cloud.google.com/tpu/docs/v6e'),
 ('07_meta_rsc_phase1','Meta RSC Phase 1','Meta','https://ai.meta.com/blog/ai-rsc/'),
 ('11_bytedance_megascale','ByteDance MegaScale','ByteDance','https://www.usenix.org/conference/nsdi24/presentation/jiang-ziheng'),
 ('21_nvidia_dgx_h100_superpod','DGX H100 SuperPOD','NVIDIA','https://docs.nvidia.com/dgx-superpod/reference-architecture-scalable-infrastructure-h100/latest/dgx-superpod-architecture.html'),
 ('22_nvidia_dgx_b200_superpod','DGX B200 SuperPOD','NVIDIA','https://docs.nvidia.com/dgx-superpod/reference-architecture-scalable-infrastructure-b200/latest/dgx-superpod-architecture.html'),
 ('24_nvidia_gb200_nvl72','GB200 NVL72 rack','NVIDIA','https://docs.nvidia.com/dgx/dgxgb200-user-guide/hardware.html'),
 ('29_frontier','Frontier','OLCF / HPE / AMD','https://docs.olcf.ornl.gov/systems/frontier_user_guide.html'),
 ('30_aurora','Aurora','ALCF / HPE / Intel','https://docs.alcf.anl.gov/aurora/'),
 ('31_nvidia_dgx_b300_1su','DGX B300 SuperPOD — 1 SU','NVIDIA','https://docs.nvidia.com/dgx-superpod/reference-architecture/scalable-infrastructure-b300-xdr/latest/dgx-superpod-architecture.html'),
]
pd.DataFrame(SOURCE_MATRIX, columns=['case_id','public_design','vendor_org','primary_source'])


## 현재 v2 엔진 직접 실행

`design_input.json`과 `reference.json`을 분리하여 읽고, `multi_arch_bom_engine_v2.js`를 직접 호출합니다. Reference는 계산 후 비교에만 사용됩니다.


In [ ]:
NODE_SCRIPT = r'''
const fs=require('fs');
const path=require('path');
const base=process.env.VAL_BASE;
const caseDir=process.env.CASE_DIR;
const v2=require(path.join(base,'engine','multi_arch_bom_engine_v2.js'));
const input=JSON.parse(fs.readFileSync(path.join(caseDir,'design_input.json'),'utf8'));
const reference=JSON.parse(fs.readFileSync(path.join(caseDir,'reference.json'),'utf8'));
const out=v2.runV2(input,reference);
console.log(JSON.stringify(out));
'''

results = {}
for case_id in CASES:
    env=os.environ.copy()
    env['VAL_BASE']=str(BASE)
    env['CASE_DIR']=str(BASE/case_id)
    cp=subprocess.run(['node','-e',NODE_SCRIPT], env=env, capture_output=True, text=True, check=True)
    results[case_id]=json.loads(cp.stdout)
print('Executed cases:', len(results))


In [ ]:
case_rows=[]
metric_rows=[]
for case_id,out in results.items():
    nv=out['numerical_validation']
    errs=[]
    for metric,d in nv['metrics'].items():
        ref=float(d['reference'])
        calc=float(d['calculated'])
        err=0.0 if ref==0 and calc==0 else (abs(calc-ref)/abs(ref)*100 if ref!=0 else np.nan)
        errs.append(err)
        metric_rows.append({
            'case_id':case_id,'metric':metric,'reference':ref,'calculated':calc,
            'error_pct_recomputed':err,'engine_error_pct':d['error_pct'],'status':d['status']
        })
    case_mape=float(np.nanmean(errs))
    case_rows.append({
        'case_id':case_id,
        'validation_level':out['validation_level'],
        'status':nv['status'],
        'coverage_pct':nv['coverage_pct'],
        'metric_count':len(errs),
        'mape_pct_recomputed':case_mape,
        'engine_mape_pct':nv['mape_pct'],
        'max_error_pct':float(np.nanmax(errs)) if errs else np.nan,
    })

case_df=pd.DataFrame(case_rows)
metric_df=pd.DataFrame(metric_rows)
case_df


In [ ]:
threshold=10.0
summary={
 'case_count':len(case_df),
 'metric_count':len(metric_df),
 'pass_cases':int((case_df['status']=='PASS').sum()),
 'coverage_min_pct':float(case_df['coverage_pct'].min()),
 'case_average_mape_pct':float(case_df['mape_pct_recomputed'].mean()),
 'metric_weighted_mape_pct':float(metric_df['error_pct_recomputed'].mean()),
 'max_case_mape_pct':float(case_df['mape_pct_recomputed'].max()),
 'max_single_metric_error_pct':float(metric_df['error_pct_recomputed'].max()),
 'all_metrics_under_10pct':bool((metric_df['error_pct_recomputed']<threshold).all()),
 'validation_levels':case_df['validation_level'].value_counts().to_dict(),
}
pd.Series(summary)


In [ ]:
display(case_df.sort_values('mape_pct_recomputed', ascending=False))
display(metric_df.sort_values('error_pct_recomputed', ascending=False).head(15))

plt.figure(figsize=(12,5))
plot_df=case_df.sort_values('mape_pct_recomputed', ascending=False)
plt.bar(plot_df['case_id'], plot_df['mape_pct_recomputed'])
plt.axhline(10, linestyle='--', linewidth=1, label='10% threshold')
plt.ylabel('Case MAPE (%)')
plt.title('2026-09-26 Public Reference Re-validation — Case MAPE')
plt.xticks(rotation=70, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
expected_case_avg=0.0042844405889298064
expected_weighted=0.004393421061836665
expected_max=0.22641509433961865

assert len(case_df)==10
assert len(metric_df)==62
assert (case_df['status']=='PASS').all()
assert (case_df['coverage_pct']==100).all()
assert (metric_df['error_pct_recomputed']<10).all()
assert abs(summary['case_average_mape_pct']-expected_case_avg)<1e-10
assert abs(summary['metric_weighted_mape_pct']-expected_weighted)<1e-10
assert abs(summary['max_single_metric_error_pct']-expected_max)<1e-10
print('SNAPSHOT CHECK: PASS')
print(json.dumps(summary, indent=2, ensure_ascii=False))


## 해석 및 한계

- 선택한 10개 case는 **10/10 PASS**, 62개 비교 metric 모두 10% 미만입니다.
- 이 notebook은 현재 GitHub의 shared engine을 다시 실행하므로 숫자를 손으로 복사한 보고서보다 재현성이 높습니다.
- 다만 이 10개는 기존 개발/회귀 ledger에 포함되어 있으므로 **strict unseen hold-out이 아닙니다**.
- 공개 문서가 제공하지 않는 cable length, exact connector SKU, patch-panel quantity, rack route 등은 scoring하지 않습니다.
- 다음 단계는 엔진을 freeze한 뒤 새로운 cross-vendor 10개 case를 reference answer를 보지 않고 처음부터 계산하는 것입니다.


In [ ]:
case_df.to_csv('/content/260926_case_summary.csv', index=False)
metric_df.to_csv('/content/260926_metric_detail.csv', index=False)
with open('/content/260926_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)
print('Exported CSV/JSON to /content/')
